mounting the drive

In [3]:
from google.colab import drive

drive.mount('/content/drive')


Mounted at /content/drive


In [5]:
import os
import glob
import pandas as pd
import numpy as np

DATA_ROOT = "/content/drive/MyDrive/Hybrid18_Final_Dataset-20260807T111200Z-1-002"

csv_files = glob.glob(
    os.path.join(DATA_ROOT, "**", "*.csv"),
    recursive=True
)

print("Total CSV files:", len(csv_files))

for file in csv_files:
    print(os.path.relpath(file, DATA_ROOT))

Total CSV files: 5
Hybrid18_Final_Dataset/1. Malware/Virus/HYBRID18_VIRUS_FINAL_WITH_PCAP_100000.csv
Hybrid18_Final_Dataset/1. Malware/spyware/HYBRID18_SPYWARE_FINAL_WITH_PCAP_100000.csv
Hybrid18_Final_Dataset/6. Credential/HYBRID18_BF_FINAL_84PLUS_FEATURES_PUBLIC_93802.csv
Hybrid18_Final_Dataset/3. Network_Based/DOS/hybrid18_flow_features_84_network.csv
Hybrid18_Final_Dataset/3. Network_Based/DDOS/hybrid18_flow_features_84_network.csv


In [6]:
datasets = {}

for file in csv_files:

    name = os.path.relpath(file, DATA_ROOT)

    datasets[name] = pd.read_csv(
        file,
        low_memory=False
    )

    print(name, datasets[name].shape)

Hybrid18_Final_Dataset/1. Malware/Virus/HYBRID18_VIRUS_FINAL_WITH_PCAP_100000.csv (100000, 161)
Hybrid18_Final_Dataset/1. Malware/spyware/HYBRID18_SPYWARE_FINAL_WITH_PCAP_100000.csv (100000, 182)
Hybrid18_Final_Dataset/6. Credential/HYBRID18_BF_FINAL_84PLUS_FEATURES_PUBLIC_93802.csv (93802, 212)
Hybrid18_Final_Dataset/3. Network_Based/DOS/hybrid18_flow_features_84_network.csv (562810, 89)
Hybrid18_Final_Dataset/3. Network_Based/DDOS/hybrid18_flow_features_84_network.csv (562810, 89)


Find common features across all 5 datasets
Create a new cell and run:

In [8]:
from itertools import combinations

# Get columns from each dataset
dataset_columns = {
    name: set(df.columns)
    for name, df in datasets.items()
}

# Find exact common features across all datasets
common_all = set.intersection(
    *dataset_columns.values()
)

print("Features common to ALL datasets:", len(common_all))

# Find common features between every pair
for (name1, cols1), (name2, cols2) in combinations(
    dataset_columns.items(), 2
):

    common = cols1.intersection(cols2)

    print("\n", "="*60)
    print("Dataset 1:", name1)
    print("Dataset 2:", name2)
    print("Common features:", len(common))
    print(sorted(common))

Features common to ALL datasets: 0

Dataset 1: Hybrid18_Final_Dataset/1. Malware/Virus/HYBRID18_VIRUS_FINAL_WITH_PCAP_100000.csv
Dataset 2: Hybrid18_Final_Dataset/1. Malware/spyware/HYBRID18_SPYWARE_FINAL_WITH_PCAP_100000.csv
Common features: 113
['attack_category', 'c2_event_id', 'c2_http_method', 'c2_ip', 'c2_network_stage', 'c2_protocol', 'c2_run_id', 'c2_seen', 'c2_src_ip', 'c2_src_port', 'c2_status', 'c2_timestamp', 'dataset_family', 'event_id', 'evidence_c2_events_csv', 'evidence_checkpoint_json', 'evidence_cleanup_events_csv', 'evidence_resource_metrics_csv', 'evidence_run_sheet_csv', 'execution_type', 'expected_network_protocol', 'global_event_id', 'http_port', 'is_attack', 'label', 'malware_stage', 'malware_type', 'net_http_method', 'net_http_uri', 'net_mapping_strategy', 'net_protocol_family', 'net_request_dst_ip', 'net_request_dst_port', 'net_request_frame_len', 'net_request_frame_number', 'net_request_frame_protocols', 'net_request_frame_time_epoch', 'net_request_http_conte

Generate a complete feature inventory
We need to see every feature, not just common ones.

In [9]:
feature_inventory = []

for dataset_name, df in datasets.items():

    for col in df.columns:

        feature_inventory.append({
            "Dataset": dataset_name,
            "Feature": col,
            "Data_Type": str(df[col].dtype),
            "Unique_Values": df[col].nunique(),
            "Missing_Percentage": round(
                df[col].isna().mean() * 100, 2
            )
        })

feature_inventory = pd.DataFrame(feature_inventory)

display(feature_inventory)

,Dataset,Feature,Data_Type,Unique_Values,Missing_Percentage
0,Hybrid18_Final_Dataset/1. Malware/Virus/HYBRID...,dataset_family,object,1,0.0
1,Hybrid18_Final_Dataset/1. Malware/Virus/HYBRID...,attack_category,object,1,0.0
2,Hybrid18_Final_Dataset/1. Malware/Virus/HYBRID...,malware_type,object,1,0.0
3,Hybrid18_Final_Dataset/1. Malware/Virus/HYBRID...,malware_stage,object,1,0.0
4,Hybrid18_Final_Dataset/1. Malware/Virus/HYBRID...,is_attack,int64,1,0.0
...,...,...,...,...,...
728,Hybrid18_Final_Dataset/3. Network_Based/DDOS/h...,IsAnomaly,int64,1,0.0
729,Hybrid18_Final_Dataset/3. Network_Based/DDOS/h...,Network Category,object,1,0.0
730,Hybrid18_Final_Dataset/3. Network_Based/DDOS/h...,Specific Attack,object,3,0.0
731,Hybrid18_Final_Dataset/3. Network_Based/DDOS/h...,Confidence,object,2,0.0


value-based feature comparison

In [ ]:
from itertools import combinations
from difflib import SequenceMatcher
import numpy as np
import pandas as pd
import re
import os

def normalize_name(name):
    return re.sub(
        r'[^a-z0-9]',
        '',
        str(name).lower()
    )

def name_similarity(a, b):
    return SequenceMatcher(
        None,
        normalize_name(a),
        normalize_name(b)
    ).ratio()


def get_distribution(series):

    values = pd.to_numeric(
        series,
        errors="coerce"
    ).dropna()

    if len(values) < 100:
        return None

    # Cast to float to avoid TypeError when numpy calculates quantiles on booleans
    return values.astype(float).quantile(
        [0, .25, .50, .75, 1]
    ).values


def compare_distribution(q1, q2):

    if q1 is None or q2 is None:
        return np.nan

    q1 = np.asarray(q1, dtype=float)
    q2 = np.asarray(q2, dtype=float)

    r1 = q1[-1] - q1[0]
    r2 = q2[-1] - q2[0]

    if r1 == 0 or r2 == 0:
        return np.nan

    q1 = (q1 - q1[0]) / r1
    q2 = (q2 - q2[0]) / r2

    return max(
        0,
        1 - np.mean(np.abs(q1 - q2))
    )


results = []

for (ds1, df1), (ds2, df2) in combinations(
    datasets.items(), 2
):

    print("Comparing:", ds1, "VS", ds2)

    for f1 in df1.columns:

        q1 = get_distribution(df1[f1])

        if q1 is None:
            continue

        for f2 in df2.columns:

            q2 = get_distribution(df2[f2])

            if q2 is None:
                continue

            value_score = compare_distribution(
                q1, q2
            )

            name_score = name_similarity(
                f1, f2
            )

            if (
                value_score >= 0.95
                or name_score >= 0.50
            ):

                results.append({

                    "Dataset_1": ds1,
                    "Feature_1": f1,

                    "Dataset_2": ds2,
                    "Feature_2": f2,

                    "Name_Similarity": round(
                        name_score, 3
                    ),

                    "Value_Similarity": round(
                        value_score, 3
                    )

                })


matching_results = pd.DataFrame(results)

if not matching_results.empty:
    matching_results = matching_results.sort_values(
        "Value_Similarity",
        ascending=False,
        na_position="last"
    )

display(matching_results)

# Ensure output directory exists
os.makedirs("/content/drive/MyDrive/Hybrid18_Project", exist_ok=True)

matching_results.to_csv(
    "/content/drive/MyDrive/Hybrid18_Project/Value_Based_Feature_Matches.csv",
    index=False
)

Comparing: Hybrid18_Final_Dataset/1. Malware/Virus/HYBRID18_VIRUS_FINAL_WITH_PCAP_100000.csv VS Hybrid18_Final_Dataset/1. Malware/spyware/HYBRID18_SPYWARE_FINAL_WITH_PCAP_100000.csv
